# Developing two Bitcoin fraud models

**Development path:** inspect labels and numerical features → establish chronological baselines → introduce directional graph dispersion → combine complementary tree models into B0 → expose individual-coordinate dispersion in Coord3 → check refinements → train and verify both final models.

| Final submission | Improvement | Public ROC-AUC |
|---|---|---:|
| `graph_lgbm_hgb.csv` — B0 | Graph-LightGBM ensemble with a raw HGB anchor | 0.94878 |
| `graph_lgbm_hgb_coord3.csv` — Coord3 | B0 with three training-selected coordinate channels | 0.94958 |


## 1. Set up environment

Use Python3.13 and install `requirements_final_models.txt`. Put `train.csv`, `test.csv`, `sample_submission.csv` and `txs_edgelist.csv` in `data/`, or change `DATA_DIR`. Select the corresponding kernel and **Restart Kernel and Run All**.

The notebook needs no previous checkpoints. Its own fingerprint-checked caches speed up repeated runs; set `USE_CACHE=False` to refit.


In [22]:
from pathlib import Path
import os, json, hashlib, time, platform, importlib.metadata, warnings
os.environ['LOKY_MAX_CPU_COUNT'] = '4'
import numpy as np
import pandas as pd
import scipy
from scipy.sparse import csr_matrix
from scipy.stats import ks_2samp
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from threadpoolctl import threadpool_limits
from IPython.display import display
import joblib

ROOT = Path.cwd()
if not (ROOT / 'data/train.csv').exists():
    candidates = [ROOT] + list(ROOT.parents)
    ROOT = next((p for p in candidates if (p / 'data/train.csv').exists()), ROOT)
DATA_DIR = ROOT / 'data' 
OUTPUT_DIR = ROOT / 'reproducibility/final_two_models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = OUTPUT_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)
USE_CACHE = True
VERIFY_EXISTING_ARTIFACTS = True 
SEEDS = [47, 48, 49]
FEATURES = [f'feat_{i}' for i in range(1, 166)]
GRAPH_COLUMNS = [
    'graph_in_dispersion', 'graph_in_deviation', 'graph_in_available',
    'graph_out_dispersion', 'graph_out_deviation', 'graph_out_available',
]
WEIGHTS = {'Slow200': 0.4, 'RF80': 0.4, 'H1': 0.2}
assert abs(sum(WEIGHTS.values()) - 1) < 1e-15
EXPECTED_VERSIONS = {
    'numpy': '2.5.4', 'pandas': '3.0.6', 'scipy': '1.18.1',
    'scikit-learn': '1.9.1', 'lightgbm': '4.7.0', 'joblib': '1.6.0',
}
VERSIONS = {k: importlib.metadata.version(k) for k in EXPECTED_VERSIONS}
if VERSIONS != EXPECTED_VERSIONS:
    raise RuntimeError('Use requirements_final_models.txt for exact reproduction. Found: ' + str(VERSIONS))

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, allow_nan=False))

START = time.perf_counter()
NEW_FITS = 0
FIT_LOG = []
DATA_HASHES = {name: file_hash(DATA_DIR / name) for name in
               ['train.csv', 'test.csv', 'sample_submission.csv', 'txs_edgelist.csv']}
prior_submissions = ROOT / 'submissions'
PRESERVED = {str(p): file_hash(p) for p in prior_submissions.glob('*.csv')} if prior_submissions.exists() else {}
display(pd.DataFrame({'package': VERSIONS.keys(), 'version': VERSIONS.values()}))


,package,version
0,numpy,2.5.4
1,pandas,3.0.6
2,scipy,1.18.1
3,scikit-learn,1.9.1
4,lightgbm,4.7.0
5,joblib,1.6.0


## 2. Understand the dataset

First separate confirmed labels (`0=licit`, `1=illicit`) from unknown transactions. Unknown rows cannot be supervised targets, but their numerical attributes and edges can describe a transaction's neighborhood.

Keep all 165 numerical features. Exclude transaction IDs, row indices, period and label from model inputs. Periods define validation boundaries; IDs map graph nodes and submission rows.

The checks below establish dataset sizes, class counts, numerical validity and official submission ordering. Explicit parsing also preserves the original numerical behavior: historical chronological HGB used standard pandas parsing, while the final HGB and graph/LightGBM calculations used round-trip parsing.


In [23]:
train = pd.read_csv(DATA_DIR / 'train.csv', float_precision='round_trip')
test = pd.read_csv(DATA_DIR / 'test.csv', float_precision='round_trip')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv', float_precision='round_trip')
edges = pd.read_csv(DATA_DIR / 'txs_edgelist.csv')
HGB_TRAIN_STANDARD = pd.read_csv(DATA_DIR / 'train.csv').set_index('txId')

assert train.txId.is_unique and test.txId.is_unique
assert set(train.txId).isdisjoint(test.txId)
assert train.time_step.between(1, 35).all() and test.time_step.between(36, 49).all()
assert 'label' not in test.columns
assert [c for c in train if c.startswith('feat_')] == FEATURES
assert [c for c in test if c.startswith('feat_')] == FEATURES
assert all(pd.api.types.is_numeric_dtype(train[f]) for f in FEATURES)
assert np.isfinite(train[FEATURES].to_numpy()).all()
assert np.isfinite(test[FEATURES].to_numpy()).all()
assert sample.columns.tolist() == ['index', 'target']
assert sample['index'].is_unique and sample['index'].notna().all()
assert test['index'].is_unique and test['index'].notna().all()
assert set(test['index']) == set(sample['index'])
test = test.set_index('index', drop=False).loc[sample['index']].reset_index(drop=True)
assert np.array_equal(test['index'], sample['index'])
confirmed = train[train.label.isin([0, 1])].copy()
confirmed['label'] = confirmed.label.astype(int)
unknown = train[~train.label.isin([0, 1])].copy()
assert len(train) == 141772 and len(test) == 15329
assert len(confirmed) == 31235 and len(unknown) == 110537
assert confirmed.label.value_counts().to_dict() == {0: 27591, 1: 3644}
R = train.set_index('txId')
display(pd.DataFrame([
    {'dataset': 'train', 'rows': len(train), 'confirmed': len(confirmed), 'unknown': len(unknown)},
    {'dataset': 'test', 'rows': len(test), 'confirmed': None, 'unknown': None},
]))


,dataset,rows,confirmed,unknown
0,train,141772,31235.0,110537.0
1,test,15329,NaN,NaN


In [24]:
period_rows = []
for period, group in train.groupby('time_step'):
    illicit = int(group.label.eq(1).sum())
    licit = int(group.label.eq(0).sum())
    known = illicit + licit
    period_rows.append(dict(time_step=int(period), total=len(group), illicit=illicit,
                            licit=licit, unknown=len(group)-known,
                            label_coverage=known/len(group),
                            illicit_rate_known=illicit/known if known else np.nan))
temporal = pd.DataFrame(period_rows)
temporal.to_csv(OUTPUT_DIR / 'temporal_eda.csv', index=False)
eda_features = train.loc[train.time_step <= 21, FEATURES]
statistics = eda_features.describe(percentiles=[.01, .25, .5, .75, .99]).T
statistics['skewness'] = eda_features.skew()
statistics['unique_count'] = eda_features.nunique()
statistics['modal_fraction'] = [eda_features[f].value_counts().iloc[0] / len(eda_features) for f in FEATURES]
statistics.to_csv(OUTPUT_DIR / 'numerical_eda_prefix21.csv')
display(temporal[['label_coverage', 'illicit_rate_known']].agg(['min','max','mean','std']))
display(statistics.loc[statistics.skewness.abs().nlargest(5).index,
                       ['mean','std','50%','99%','skewness','modal_fraction']])
print('Skew/concentration motivate robust graph geometry; all original model features remain intact.')


,label_coverage,illicit_rate_known
min,0.112061,0.004251
max,0.299118,0.359703
mean,0.216237,0.133711
std,0.047544,0.101758


,mean,std,50%,99%,skewness,modal_fraction
feat_72,-0.014240,0.285097,-0.017640,0.003641,176.233580,0.718756
feat_73,-0.011308,0.280309,-0.015071,0.005837,175.418030,0.704027
feat_71,-0.025701,0.288622,-0.030026,-0.009995,150.148858,0.704027
feat_162,-0.048168,1.076910,-0.131155,0.751985,137.451444,0.566061
feat_14,0.001649,1.192828,-0.042955,0.341693,135.810088,0.781652


Skew/concentration motivate robust graph geometry; all original model features remain intact.


### Findings 

The counts show substantial unknown-label coverage and an imbalanced confirmed target. Numerical summaries reveal skewed and concentrated features, so preserve every original input and avoid treating descriptive outliers as errors. Use robust normalization only to make neighbor-distance calculations comparable.


## 3. Start with individual supervised tree models

I began with the original 165 numerical features and no graph inputs. HGB and LightGBM both build successive trees, but their implementations and stopping rules differ. This establishes whether the raw data already provide useful discrimination.

I first used HGB with 31 leaves and basic LightGBM with 31 leaves, learning rate 0.1 and 100 rounds. I then examined three limited changes:

1. **H1:** reduce HGB's leaf limit from 31 to 15, keeping its learning rate 0.1 and maximum100 iterations.
2. **Slow200:** reduce LightGBM's learning rate to0.05 and increase rounds to 200, keeping31 leaves.
3. **RF80:** retain learning rate 0.1 and 100 rounds but sample 80% of rows and features. It remains gradient boosting, not a random forest.

The code below trains or verifies each configuration on the same fitting rows and evaluates the same chronological observations using seed 47. These are controlled parameter comparisons, not an exhaustive grid search.


In [25]:
BASE_PARAMETERS = {'Slow200': {'boosting_type': 'gbdt',
             'class_weight': None,
             'colsample_bytree': 1.0,
             'importance_type': 'split',
             'learning_rate': 0.05,
             'max_depth': -1,
             'min_child_samples': 20,
             'min_child_weight': 0.001,
             'min_split_gain': 0.0,
             'n_estimators': 200,
             'n_jobs': 4,
             'num_leaves': 31,
             'objective': 'binary',
             'random_state': 47,
             'reg_alpha': 0.0,
             'reg_lambda': 0.0,
             'subsample': 1.0,
             'subsample_for_bin': 200000,
             'subsample_freq': 0,
             'deterministic': True,
             'force_col_wise': True,
             'verbosity': -1},
 'RF80': {'boosting_type': 'gbdt',
          'class_weight': None,
          'colsample_bytree': 0.8,
          'importance_type': 'split',
          'learning_rate': 0.1,
          'max_depth': -1,
          'min_child_samples': 20,
          'min_child_weight': 0.001,
          'min_split_gain': 0.0,
          'n_estimators': 100,
          'n_jobs': 4,
          'num_leaves': 31,
          'objective': 'binary',
          'random_state': 47,
          'reg_alpha': 0.0,
          'reg_lambda': 0.0,
          'subsample': 0.8,
          'subsample_for_bin': 200000,
          'subsample_freq': 1,
          'deterministic': True,
          'force_col_wise': True,
          'verbosity': -1},
 'H1': {'categorical_features': 'from_dtype',
        'class_weight': None,
        'early_stopping': 'auto',
        'interaction_cst': None,
        'l2_regularization': 0.0,
        'learning_rate': 0.1,
        'loss': 'log_loss',
        'max_bins': 255,
        'max_depth': None,
        'max_features': 1.0,
        'max_iter': 100,
        'max_leaf_nodes': 15,
        'min_samples_leaf': 20,
        'monotonic_cst': None,
        'n_iter_no_change': 10,
        'random_state': 47,
        'scoring': 'loss',
        'tol': 1e-07,
        'validation_fraction': 0.1,
        'verbose': 0,
        'warm_start': False}}

def parameters(family, seed):
    if family == 'LGB100':
        p = dict(BASE_PARAMETERS['Slow200'], learning_rate=.1, n_estimators=100)
    elif family == 'HGB31':
        p = dict(BASE_PARAMETERS['H1'], max_leaf_nodes=31)
    else:
        p = dict(BASE_PARAMETERS[family])
    p['random_state'] = seed
    return p

save_json(OUTPUT_DIR / 'frozen_parameters.json', {
    'base_parameters': BASE_PARAMETERS, 'seeds': SEEDS, 'weights': WEIGHTS,
    'data_hashes': DATA_HASHES, 'versions': VERSIONS,
})
display(pd.DataFrame([
    {'model':'HGB31 reference','leaves':31,'learning_rate':.1,'maximum_rounds':100,'sampling':'none'},
    {'model':'H1 retained','leaves':15,'learning_rate':.1,'maximum_rounds':100,'sampling':'none'},
    {'model':'LGB100 reference','leaves':31,'learning_rate':.1,'maximum_rounds':100,'sampling':'none'},
    {'model':'Slow200 retained','leaves':31,'learning_rate':.05,'maximum_rounds':200,'sampling':'none'},
    {'model':'RF80 retained','leaves':31,'learning_rate':.1,'maximum_rounds':100,'sampling':'80% rows/features'},
]))

def fit_model(name, family, seed, frame, labels, ids):
    global NEW_FITS
    params = parameters(family, seed)
    labels = np.asarray(labels, dtype=np.int64)
    assert len(frame) == len(labels) == len(ids) and set(labels) == {0,1}
    stamp = hashlib.sha256()
    stamp.update(np.asarray(frame, dtype=np.float64).tobytes())
    stamp.update(labels.tobytes())
    stamp.update(np.asarray(ids, dtype=np.int64).tobytes())
    stamp.update(json.dumps({'columns':frame.columns.tolist(), 'params':params,
                             'versions':VERSIONS}, sort_keys=True).encode())
    fingerprint = stamp.hexdigest()
    path = MODEL_DIR / (name + '.joblib')
    metadata_path = MODEL_DIR / (name + '.json')
    if USE_CACHE and path.exists() and metadata_path.exists():
        metadata = json.loads(metadata_path.read_text())
        assert metadata['fingerprint'] == fingerprint and metadata['sha256'] == file_hash(path)
        model = joblib.load(path)
        reused = True
    else:
        tic = time.perf_counter()
        cls = HistGradientBoostingClassifier if family in ['H1','HGB31'] else LGBMClassifier
        with threadpool_limits(limits=4):
            model = cls(**params).fit(frame, labels)
        joblib.dump(model, path)
        NEW_FITS += 1
        metadata = {'fingerprint':fingerprint, 'sha256':file_hash(path),
                    'parameters':params, 'features':frame.columns.tolist(),
                    'fit_ids':list(map(int,ids)), 'n':len(labels), 'illicit':int(labels.sum()),
                    'fit_seconds':time.perf_counter()-tic,
                    'rounds':int(model.n_iter_) if family in ['H1','HGB31'] else model.booster_.current_iteration()}
        save_json(metadata_path, metadata)
        reused = False
    assert model.get_params() == params and list(model.classes_) == [0,1]
    assert model.feature_names_in_.tolist() == frame.columns.tolist()
    FIT_LOG.append({'name':name,'reused':reused,'fit_seconds':metadata['fit_seconds']})
    return model

def probability(model, frame):
    assert frame.columns.tolist() == model.feature_names_in_.tolist()
    with threadpool_limits(limits=4):
        result = model.predict_proba(frame)[:, list(model.classes_).index(1)]
    assert np.isfinite(result).all() and ((result >= 0) & (result <= 1)).all()
    return result

def metrics(y, score):
    y = np.asarray(y, dtype=int)
    tn, fp, fn, tp = confusion_matrix(y, np.asarray(score)>=.5, labels=[0,1]).ravel()
    return {'auc':float(roc_auc_score(y,score)) if len(np.unique(y))==2 else np.nan,
            'ap':float(average_precision_score(y,score)) if y.sum() else np.nan,
            'n':len(y),'illicit':int(y.sum()),'tp':int(tp),'fp':int(fp),'tn':int(tn),'fn':int(fn),
            'recall':float(tp/(tp+fn)) if tp+fn else np.nan,
            'precision':float(tp/(tp+fp)) if tp+fp else 0.}


,model,leaves,learning_rate,maximum_rounds,sampling
0,HGB31 reference,31,0.10,100,none
1,H1 retained,15,0.10,100,none
2,LGB100 reference,31,0.10,100,none
3,Slow200 retained,31,0.05,200,none
4,RF80 retained,31,0.10,100,80% rows/features


In [26]:
baseline_rows = []
for cutoff in [21, 28]:
    fitting = confirmed[confirmed.time_step <= cutoff]
    evaluation = train[train.time_step > cutoff]
    ids = fitting.txId.tolist()
    raw_fit = fitting.set_index('txId')[FEATURES]
    raw_eval = evaluation.set_index('txId')[FEATURES]
    for family, name in [('HGB31','HGB31'), ('LGB100','LGB100'),
                         ('H1','H1'), ('Slow200','RawSlow'), ('RF80','RawRF')]:
        hgb = family in ['H1', 'HGB31']
        model_name = f'prefix{cutoff}_seed47_reference_{family}' if family in ['HGB31','LGB100'] else f'prefix{cutoff}_seed47_{name}'
        model = fit_model(model_name, family, 47,
                          HGB_TRAIN_STANDARD.loc[ids,FEATURES] if hgb else raw_fit,
                          fitting.label, ids)
        score = probability(model, HGB_TRAIN_STANDARD.loc[evaluation.txId,FEATURES] if hgb else raw_eval)
        windows = [('A',22,28),('Long',29,35)] if cutoff == 21 else [('B',29,35)]
        for window, low, high in windows:
            mask = (evaluation.time_step.between(low,high) & evaluation.label.isin([0,1])).to_numpy()
            baseline_rows.append(dict(window=window,model=name,seed=47,
                                      **metrics(evaluation.loc[mask,'label'],score[mask])))
references = pd.DataFrame(baseline_rows)
references.to_csv(OUTPUT_DIR/'seed47_baseline_parameter_comparison.csv',index=False)
display(references[['window','model','seed','auc','ap']])


,window,model,seed,auc,ap
0,A,HGB31,47,0.946797,0.876589
1,Long,HGB31,47,0.824075,0.753768
2,A,LGB100,47,0.957212,0.884304
3,Long,LGB100,47,0.873957,0.777369
4,A,H1,47,0.952543,0.880633
5,Long,H1,47,0.851412,0.761646
6,A,RawSlow,47,0.945735,0.877552
7,Long,RawSlow,47,0.839684,0.761020
8,A,RawRF,47,0.952118,0.879546
9,Long,RawRF,47,0.860230,0.773582


### What the individual-model comparison tells us

H1 improves seed 47 Long AUC over HGB31 (.824075→.851412), supporting a smaller HGB component. However, basic LightGBM has stronger single-seed Long AUC (.873957) than raw Slow200/RF80. 

The next question is whether combining different tree predictions and adding graph information improves a **matched ensemble**. 


## 5. Build the raw-feature reference ensemble

I combined Slow200, RF80 and H1 probabilities as 40%,40%,20%. H1 supplies a smaller tree model, while the two LightGBM branches use different learning/sampling schedules. I repeat all three seeds and average final probabilities rather than selecting the best seed.

This is still a raw-feature model. It is the reference for deciding whether graph features help, with model settings, seeds, fitting rows and weights held constant. Raw LightGBM components will also supply the missing-neighbour fallback later.


In [27]:
def mean_predictions(seed_predictions, evaluation, columns):
    mean = seed_predictions.groupby('txId',sort=False)[columns].mean()
    mean = mean.loc[evaluation.txId].reset_index()
    mean['time_step'] = evaluation.time_step.to_numpy()
    if 'label' in evaluation:
        mean['label'] = evaluation.label.to_numpy()
    return mean

def fit_raw_stage(cutoff, evaluation, final=False):
    fitting_context = train[train.time_step<=cutoff].copy()
    fitting = fitting_context[fitting_context.label.isin([0,1])].copy()
    fitting['label'] = fitting.label.astype(int)
    assert evaluation.time_step.min()>cutoff
    assert set(fitting.txId).isdisjoint(evaluation.txId)
    ids = fitting.txId.tolist()
    raw_fit = fitting.set_index('txId')[FEATURES]
    raw_eval = evaluation.set_index('txId')[FEATURES]
    hfit = raw_fit if final else HGB_TRAIN_STANDARD.loc[ids,FEATURES]
    heval = raw_eval if final else HGB_TRAIN_STANDARD.loc[evaluation.txId,FEATURES]
    outputs, models = [], {}
    for seed in SEEDS:
        row = evaluation[['txId','time_step']].copy().reset_index(drop=True)
        if 'label' in evaluation:
            row['label'] = evaluation.label.to_numpy()
        row['seed'] = seed
        for name,family,frame,target in [('RawSlow','Slow200',raw_fit,raw_eval),
                                       ('RawRF','RF80',raw_fit,raw_eval),
                                       ('H1','H1',hfit,heval)]:
            model = fit_model(f'prefix{cutoff}_seed{seed}_{name}',family,seed,
                              frame,fitting.label,ids)
            models[(name,seed)] = model
            row[name] = probability(model,target)
        row['RawBlend'] = .4*row.RawSlow+.4*row.RawRF+.2*row.H1
        outputs.append(row)
    predictions = pd.concat(outputs,ignore_index=True)
    return dict(cutoff=cutoff,fitting_context=fitting_context,fitting=fitting,
                evaluation=evaluation,raw_fit=raw_fit,raw_eval=raw_eval,
                hfit=hfit,heval=heval,models=models,seeds=predictions,
                mean=mean_predictions(predictions,evaluation,['RawSlow','RawRF','H1','RawBlend']))

WINDOWS = [('A',21,22,28),('B',28,29,35),('Long',21,29,35)]
def evaluate_stages(stages, model_names):
    rows = []
    for window,cutoff,low,high in WINDOWS:
        group = stages[cutoff]['mean']
        group = group[group.time_step.between(low,high)&group.label.isin([0,1])]
        for name in model_names:
            rows.append(dict(window=window,model=name,**metrics(group.label,group[name])))
    return pd.DataFrame(rows)

raw_stages = {}
for cutoff in [21,28]:
    raw_stages[cutoff] = fit_raw_stage(cutoff,train[train.time_step>cutoff].copy())
raw_validation = evaluate_stages(raw_stages,['RawSlow','RawRF','H1','RawBlend'])
raw_validation.to_csv(OUTPUT_DIR/'raw_reference_metrics.csv',index=False)
display(raw_validation[raw_validation.model=='RawBlend'][['window','model','auc','ap','tp','fp','fn','recall','precision']])


,window,model,auc,ap,tp,fp,fn,recall,precision
3,A,RawBlend,0.953598,0.882735,527,24,144,0.785395,0.956443
7,B,RawBlend,0.996758,0.987500,1003,17,99,0.910163,0.983333
11,Long,RawBlend,0.851988,0.767814,673,26,429,0.610708,0.962804


## 6. Add directional graph information to create B0

The next hypothesis was that relationships to connected transactions could add information beyond the focal transaction's165 features. For each incoming/outgoing direction, I calculate aggregate spread, focal deviation and an availability flag: six additional inputs in total. The original 165 values remain raw model inputs; only graph calculations use confirmed-training median/IQR and asinh normalization.

I fit graph-enhanced Slow200/RF80 on 171 inputs, keep raw H1, and preserve40/40/20 weights. When both directions are unavailable, the graph branches use their corresponding raw-model scores. This creates **B0**. 


In [28]:
def fit_geometry(fitting):
    q = np.quantile(fitting[FEATURES].to_numpy(), [.25,.5,.75], axis=0)
    return {'median':q[1].tolist(),
            'scale':np.where(q[2]-q[0]<=1e-8, 1., q[2]-q[0]).tolist(),
            'fit_ids':fitting.txId.astype(int).tolist()}

def adjacency(frame):
    index = pd.Index(frame.txId)
    source = index.get_indexer(edges.txId1)
    target = index.get_indexer(edges.txId2)
    keep = (source >= 0) & (target >= 0)
    source, target = source[keep], target[keep]
    periods = frame.time_step.to_numpy()
    assert np.array_equal(periods[source], periods[target])
    assert not np.any(source == target)
    a = csr_matrix((np.ones(len(source)), (source,target)), shape=(len(frame),len(frame)))
    assert a.nnz == len(source), 'Unexpected duplicate edges: investigate rather than silently aggregate.'
    return a.T.tocsr(), a

def normalized_features(frame, geometry):
    return np.arcsinh((frame[FEATURES].to_numpy()-np.asarray(geometry['median'])) /
                     np.asarray(geometry['scale']))

def aggregate_graph_features(z, incoming, outgoing):
    n, k = z.shape
    out = np.empty((n,6), dtype=float)
    for direction, a in enumerate([incoming,outgoing]):
        degree = np.asarray(a.sum(axis=1)).ravel()
        denominator = np.maximum(degree,1)
        variance, distance = np.zeros(n), np.zeros(n)
        for j in range(0,k,16):
            block = z[:,j:j+16]
            mean = np.asarray(a @ block)/denominator[:,None]
            second = np.asarray(a @ (block*block))/denominator[:,None]
            variance += np.maximum(second-mean*mean,0).sum(axis=1)
            distance += ((block-mean)**2).sum(axis=1)
        spread = np.sqrt(variance/k)
        deviation = np.sqrt(distance/k)/(1+spread)
        available = degree>0
        spread[~available], deviation[~available] = np.nan, np.nan
        out[:,3*direction:3*direction+3] = np.column_stack([spread,deviation,available])
    return out

def coordinate_graph_features(z, incoming, outgoing, operands):
    columns, values = [], []
    for direction, a in [('in',incoming),('out',outgoing)]:
        degree = np.asarray(a.sum(axis=1)).ravel()
        denominator = np.maximum(degree,1)[:,None]
        mean = np.asarray(a@z)/denominator
        second = np.asarray(a@(z*z))/denominator
        spread = np.sqrt(np.maximum(second-mean*mean,0))
        deviation = np.abs(z-mean)/(1+spread)
        spread[degree==0], deviation[degree==0] = np.nan,np.nan
        for j, operand in enumerate(operands):
            values.extend([spread[:,j],deviation[:,j]])
            columns.extend([f'coord_{direction}_std_{operand}', f'coord_{direction}_dev_{operand}'])
    return np.column_stack(values), columns

def graph_table(frame, geometry, operands=None):
    incoming, outgoing = adjacency(frame)
    z = normalized_features(frame,geometry)
    aggregate = aggregate_graph_features(z,incoming,outgoing)
    result = pd.DataFrame(aggregate,index=frame.txId,columns=GRAPH_COLUMNS)
    if operands is not None:
        positions = [FEATURES.index(f) for f in operands]
        extra, columns = coordinate_graph_features(z[:,positions],incoming,outgoing,operands)
        result = result.join(pd.DataFrame(extra,index=frame.txId,columns=columns))
    return result

def no_neighbors(graph):
    return (graph.graph_in_available.to_numpy()==0) & (graph.graph_out_available.to_numpy()==0)

def apply_fallback(graph_score, raw_score, graph):
    result = np.asarray(graph_score).copy()
    mask = no_neighbors(graph)
    result[mask] = np.asarray(raw_score)[mask]
    assert np.array_equal(result[mask],np.asarray(raw_score)[mask])
    return result

toy = np.array([[-1.],[1.],[0.]])
a = csr_matrix((np.ones(2),([0,1],[2,2])),shape=(3,3))
g = aggregate_graph_features(toy,a.T.tocsr(),a)
c, columns = coordinate_graph_features(toy,a.T.tocsr(),a,['toy'])
assert g[2,0]==1 and g[2,1]==0 and g[0,5]==1 and g[0,2]==0
assert c[2,0]==1 and c[2,1]==0 and np.isnan(c[0,0])
empty = pd.DataFrame(np.column_stack([np.full((3,2),np.nan),np.zeros(3),
                                    np.full((3,2),np.nan),np.zeros(3)]),columns=GRAPH_COLUMNS)
assert np.array_equal(apply_fallback([.9,.8,.7],[.1,.2,.3],empty),[.1,.2,.3])
print('Graph direction, spread and exact fallback fixtures passed.')


Graph direction, spread and exact fallback fixtures passed.


### Verify graph availability and fallback

An edge is usable only if both endpoint feature rows exist. The coverage table checks the historical and test snapshots separately; test numerical distributions or labels do not determine feature selection. Direction/spread fixtures above verify the formulas. Historical snapshots contain no isolates, but approximately25% of test transactions lack neighbours, making exact raw fallback essential.


In [29]:
coverage_rows = []
for name, frame in [('train', train), ('test', test)]:
    incoming, outgoing = adjacency(frame)
    degree_in = np.asarray(incoming.sum(axis=1)).ravel()
    degree_out = np.asarray(outgoing.sum(axis=1)).ravel()
    isolates = (degree_in==0) & (degree_out==0)
    coverage_rows.append(dict(dataset=name,nodes=len(frame),available_edges=int(outgoing.nnz),
                              isolates=int(isolates.sum()),isolate_fraction=float(isolates.mean())))
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(OUTPUT_DIR/'graph_coverage.csv',index=False)
display(coverage)


,dataset,nodes,available_edges,isolates,isolate_fraction
0,train,141772,163194,0,0.000000
1,test,15329,12724,3891,0.253833


In [30]:
def add_b0_graph(raw_stage):
    cutoff = raw_stage['cutoff']
    fitting = raw_stage['fitting']
    context, evaluation = raw_stage['fitting_context'], raw_stage['evaluation']
    ids = fitting.txId.tolist()
    geometry = fit_geometry(fitting)
    save_json(OUTPUT_DIR/f'geometry_prefix{cutoff}.json',geometry)
    gf = graph_table(context,geometry)
    gv = graph_table(evaluation,geometry)
    graph_fit = raw_stage['raw_fit'].join(gf.loc[ids])
    graph_eval = raw_stage['raw_eval'].join(gv.loc[evaluation.txId])
    models, outputs = dict(raw_stage['models']), []
    for seed in SEEDS:
        row = raw_stage['seeds'][raw_stage['seeds'].seed==seed].copy()
        scores = {}
        for name,family in [('GraphSlow','Slow200'),('GraphRF','RF80')]:
            model = fit_model(f'prefix{cutoff}_seed{seed}_{name}',family,seed,
                              graph_fit,fitting.label,ids)
            models[(name,seed)] = model
            scores[name] = probability(model,graph_eval)
            row[name] = scores[name]
        row['B0'] = (.4*apply_fallback(scores['GraphSlow'],row.RawSlow.to_numpy(),gv)
                     +.4*apply_fallback(scores['GraphRF'],row.RawRF.to_numpy(),gv)
                     +.2*row.H1.to_numpy())
        row['isolated'] = no_neighbors(gv)
        outputs.append(row)
    predictions = pd.concat(outputs,ignore_index=True)
    mean = mean_predictions(predictions,evaluation,['RawSlow','RawRF','H1','RawBlend','B0'])
    mean['isolated'] = no_neighbors(gv)
    return dict(raw_stage,models=models,seeds=predictions,mean=mean,
                geometry=geometry,graph_fit=gf,graph_eval=gv)

b0_stages = {cutoff:add_b0_graph(stage) for cutoff,stage in raw_stages.items()}
b0_validation = evaluate_stages(b0_stages,['RawBlend','B0'])
b0_validation.to_csv(OUTPUT_DIR/'b0_vs_raw_metrics.csv',index=False)
display(b0_validation[['window','model','auc','ap','tp','fp','fn','recall','precision']])


,window,model,auc,ap,tp,fp,fn,recall,precision
0,A,RawBlend,0.953598,0.882735,527,24,144,0.785395,0.956443
1,A,B0,0.954628,0.883092,526,27,145,0.783905,0.951175
2,B,RawBlend,0.996758,0.987500,1003,17,99,0.910163,0.983333
3,B,B0,0.996899,0.988396,1008,18,94,0.914701,0.982456
4,Long,RawBlend,0.851988,0.767814,673,26,429,0.610708,0.962804
5,Long,B0,0.855688,0.768448,675,24,427,0.612523,0.965665


### B0 decision

B0 raises Long AUC from the matched raw blend's 0.851988 to 0.855688 and AP from 0.767814 to 0.768448, with improvements also in A/B. This is a modest measured contribution from graph features. I retain B0 as the first final model.


## 7. Refine B0 into Coord3

The six graph statistics summarize all 165 coordinates. A potentially useful difference in one coordinate can be obscured by that aggregation. I therefore test three explicit coordinate channels rather than adding many arbitrary interactions.

For each chosen coordinate and direction, add neighbour population standard deviation and `abs(focal-neighbour_mean)/(1+std)`. Three coordinates × two directions × two statistics add 12 inputs, so only the graph-LightGBM branches grow from 171 to 183. H1, weights, seeds, original inputs and fallback remain unchanged.

Select coordinates within each fitting prefix using original GraphSlow200 seed 47 gain among the original features. Eligibility requires positive gain, at least five distinct values, IQR>1e−8 and KS≤.10 comparing periods 1–14 against15–cutoff. Select the top three gains, tie by original order. No outer validation labels or test distributions select operands. If fewer than three qualify, stop rather than weaken the rule.


In [31]:
def select_coordinates(fitting, original_graph_slow_model, cutoff):
    assert fitting.time_step.max() <= cutoff and cutoff > 14
    gain = original_graph_slow_model.booster_.feature_importance(importance_type='gain')[:165]
    rows = []
    for j, feature in enumerate(FEATURES):
        values = fitting[feature]
        ks = float(ks_2samp(fitting.loc[fitting.time_step<=14,feature],
                           fitting.loc[fitting.time_step>14,feature], method='asymp').statistic)
        iqr = float(values.quantile(.75)-values.quantile(.25))
        unique = int(values.nunique())
        eligible = ks<=.10 and iqr>1e-8 and unique>=5 and gain[j]>0
        rows.append({'feature':feature,'original_order':j,'gain':float(gain[j]),
                     'training_KS':ks,'IQR':iqr,'unique':unique,'eligible':bool(eligible)})
    ranking = pd.DataFrame(rows)
    selected = ranking[ranking.eligible].sort_values(['gain','original_order'],ascending=[False,True]).head(3).feature.tolist()
    assert len(selected)==3, 'Fewer than three eligible coordinates; do not weaken the frozen rule.'
    ranking['selected'] = ranking.feature.isin(selected)
    ranking.to_csv(OUTPUT_DIR/f'coordinate_ranking_prefix{cutoff}.csv',index=False)
    return selected


In [32]:
def add_coordinate_channels(b0_stage):
    cutoff = b0_stage['cutoff']
    fitting, evaluation = b0_stage['fitting'], b0_stage['evaluation']
    ids = fitting.txId.tolist()
    operands = select_coordinates(fitting,b0_stage['models'][('GraphSlow',47)],cutoff)
    cf = graph_table(b0_stage['fitting_context'],b0_stage['geometry'],operands)
    cv = graph_table(evaluation,b0_stage['geometry'],operands)
    assert np.allclose(cf[GRAPH_COLUMNS],b0_stage['graph_fit'],equal_nan=True,rtol=0,atol=0)
    coordinate_fit = b0_stage['raw_fit'].join(cf.loc[ids])
    coordinate_eval = b0_stage['raw_eval'].join(cv.loc[evaluation.txId])
    outputs = []
    for seed in SEEDS:
        row = b0_stage['seeds'][b0_stage['seeds'].seed==seed].copy()
        scores = {}
        for family in ['Slow200','RF80']:
            model = fit_model(f'prefix{cutoff}_seed{seed}_Coord_{family}',family,seed,
                              coordinate_fit,fitting.label,ids)
            scores[family] = probability(model,coordinate_eval)
        row['Coord3'] = .4*scores['Slow200']+.4*scores['RF80']+.2*row.H1.to_numpy()
        outputs.append(row)
    predictions = pd.concat(outputs,ignore_index=True)
    mean = mean_predictions(predictions,evaluation,['RawSlow','RawRF','H1','RawBlend','B0','Coord3'])
    mean['isolated'] = no_neighbors(cv)
    mask = mean.isolated.to_numpy()
    mean.loc[mask,'Coord3'] = mean.loc[mask,'B0']
    assert np.array_equal(mean.loc[mask,'Coord3'],mean.loc[mask,'B0'])
    predictions.to_csv(OUTPUT_DIR/f'seed_predictions_prefix{cutoff}.csv',index=False)
    mean.to_csv(OUTPUT_DIR/f'mean_predictions_prefix{cutoff}.csv',index=False)
    save_json(OUTPUT_DIR/f'operands_prefix{cutoff}.json',{'selected':operands})
    return dict(b0_stage,seeds=predictions,mean=mean,operands=operands)

def fit_stage(cutoff, evaluation, final=False):
    raw_stage = fit_raw_stage(cutoff,evaluation,final)
    return add_coordinate_channels(add_b0_graph(raw_stage))

stages = {cutoff:add_coordinate_channels(stage) for cutoff,stage in b0_stages.items()}
validation = evaluate_stages(stages,['RawSlow','RawRF','H1','RawBlend','B0','Coord3'])
validation.to_csv(OUTPUT_DIR/'chronological_metrics.csv',index=False)
per_period = []
for window,cutoff,low,high in WINDOWS:
    group = stages[cutoff]['mean']
    group = group[group.time_step.between(low,high)&group.label.isin([0,1])]
    for name in ['RawSlow','RawRF','H1','RawBlend','B0','Coord3']:
        for period,part in group.groupby('time_step'):
            per_period.append(dict(window=window,model=name,time_step=int(period),**metrics(part.label,part[name])))
per_period = pd.DataFrame(per_period)
per_period.to_csv(OUTPUT_DIR/'per_period_metrics.csv',index=False)
display(pd.DataFrame([{'cutoff':cutoff,'coordinates':', '.join(stage['operands'])} for cutoff,stage in stages.items()]))


,cutoff,coordinates
0,21,"feat_47, feat_41, feat_163"
1,28,"feat_53, feat_90, feat_163"


### Coord3 decision: inspect the matched improvement

Compare Coord3 directly with B0 on identical rows and the same three-seed averaging. Long AUC increases 0.855688 → 0.869111 and AP 0.768448 → 0.775802; A improves and B is preserved.


In [33]:
comparison_rows = []
for window in ['A','B','Long']:
    b0 = validation[(validation.window==window) & (validation.model=='B0')].iloc[0]
    coord = validation[(validation.window==window) & (validation.model=='Coord3')].iloc[0]
    comparison_rows.append(dict(window=window,B0_AUC=b0.auc,Coord3_AUC=coord.auc,
                                AUC_change=coord.auc-b0.auc,B0_AP=b0.ap,Coord3_AP=coord.ap))
display(pd.DataFrame(comparison_rows))

display(validation[validation.model.isin(['B0','Coord3'])][['window','model','tp','fp','tn','fn','recall','precision']])


,window,B0_AUC,Coord3_AUC,AUC_change,B0_AP,Coord3_AP
0,A,0.954628,0.957081,0.002453,0.883092,0.884828
1,B,0.996899,0.996957,0.000058,0.988396,0.988651
2,Long,0.855688,0.869111,0.013423,0.768448,0.775802


,window,model,tp,fp,tn,fn,recall,precision
4,A,B0,526,27,4979,145,0.783905,0.951175
5,A,Coord3,526,27,4979,145,0.783905,0.951175
10,B,B0,1008,18,4908,94,0.914701,0.982456
11,B,Coord3,998,12,4914,104,0.905626,0.988119
16,Long,B0,675,24,4902,427,0.612523,0.965665
17,Long,Coord3,675,25,4901,427,0.612523,0.964286


## 8. Reproduce the five-coordinate and regularization experiments

**Five coordinates:** keep the training-only ranking/eligibility rule but use the top five rather than top three. Refit only the two graph branches at both cutoffs and seeds47/48/49; reuse H1, weights and fallback.

**Nested regularization:** compare unchanged settings, `reg_lambda=1`, or `num_leaves=15` in both graph branches. For outer cutoff21, inner fitting is1–14 and assessment15–21; for cutoff28, fitting is1–21 and assessment22–28. Fit normalization and operand selection within each inner training prefix. For inner cutoff14, KS compares1–7 versus8–14 because the usual1–14 versus15+ split is unavailable; all thresholds and the top-three rule remain unchanged. Reject insufficient support rather than relax criteria.

Select settings by complete-ensemble inner AUC, then AP, then prefer the unchanged model on exact ties, using fixed seed47. Refit only those training-selected settings on the full outer prefix for seeds47/48/49. No outer labels select settings. The original vectorized inner-graph calculation is retained for numerical reproduction. Final inference/fallback stays identical.

Models, scopes, predictions and metrics are cached under `step8_reproduced_experiments/` within this notebook's output directory. First execution fits missing models; `USE_CACHE=False` forces fresh fits. Neither experiment alters the later final-submission recipe.


In [34]:
STEP8_DIR = OUTPUT_DIR / 'step8_reproduced_experiments'
STEP8_DIR.mkdir(exist_ok=True)
_s8_new_fits = 0
_s8_settings = {'C0': {}, 'L2': {'reg_lambda': 1.0}, 'Leaves15': {'num_leaves': 15}}

def _s8_fit(name, family, seed, frame, labels, ids, changes=None):
    global _s8_new_fits
    params = parameters(family, seed)
    params.update(changes or {})
    labels = np.asarray(labels, dtype=np.int64)
    stamp = hashlib.sha256()
    stamp.update(np.asarray(frame, dtype=np.float64).tobytes())
    stamp.update(labels.tobytes())
    stamp.update(np.asarray(ids, dtype=np.int64).tobytes())
    stamp.update(json.dumps({'columns': frame.columns.tolist(), 'params': params,
                             'versions': VERSIONS}, sort_keys=True).encode())
    fingerprint = stamp.hexdigest()
    path, metadata_path = STEP8_DIR / (name + '.joblib'), STEP8_DIR / (name + '.json')
    if USE_CACHE and path.exists() and metadata_path.exists():
        metadata = json.loads(metadata_path.read_text())
        assert metadata['fingerprint'] == fingerprint and metadata['sha256'] == file_hash(path)
        model = joblib.load(path)
    else:
        cls = HistGradientBoostingClassifier if family == 'H1' else LGBMClassifier
        tic = time.perf_counter()
        with threadpool_limits(limits=4):
            model = cls(**params).fit(frame, labels)
        joblib.dump(model, path)
        metadata = dict(fingerprint=fingerprint, sha256=file_hash(path), params=params,
                        features=frame.columns.tolist(), fit_ids=list(map(int, ids)),
                        fit_seconds=time.perf_counter()-tic)
        save_json(metadata_path, metadata)
        _s8_new_fits += 1
    assert list(model.classes_) == [0, 1] and model.get_params() == params
    assert model.feature_names_in_.tolist() == frame.columns.tolist()
    return model

def _s8_outer_scores(cutoff, operands, setting, name):
    stage = stages[cutoff]
    fitting, evaluation = stage['fitting'], stage['evaluation']
    ids = fitting.txId.tolist()
    gf = graph_table(stage['fitting_context'], stage['geometry'], operands)
    gv = graph_table(evaluation, stage['geometry'], operands)
    xf = stage['raw_fit'].join(gf.loc[ids])
    xv = stage['raw_eval'].join(gv.loc[evaluation.txId])
    rows = []
    for seed in SEEDS:
        row = stage['seeds'][stage['seeds'].seed == seed][
            ['txId', 'time_step', 'label', 'seed', 'H1']].copy()
        if name == 'Nested regularization' and setting == 'C0':
            row['score'] = stage['seeds'].loc[stage['seeds'].seed == seed, 'Coord3'].to_numpy()
        else:
            scores = {}
            for family in ['Slow200', 'RF80']:
                model = _s8_fit(f'{name.replace(" ", "_")}_prefix{cutoff}_seed{seed}_{family}',
                                family, seed, xf, fitting.label, ids, _s8_settings[setting])
                scores[family] = probability(model, xv)
            row['score'] = .4*scores['Slow200'] + .4*scores['RF80'] + .2*row.H1.to_numpy()
        rows.append(row[['txId', 'time_step', 'label', 'seed', 'score']])
    seed_predictions = pd.concat(rows, ignore_index=True)
    mean = mean_predictions(seed_predictions, evaluation, ['score'])
    isolated = no_neighbors(gv)
    reference = stage['mean'].set_index('txId').loc[evaluation.txId, 'B0'].to_numpy()
    mean.loc[isolated, 'score'] = reference[isolated]
    assert np.array_equal(mean.loc[isolated, 'score'], reference[isolated])
    seed_predictions.to_csv(STEP8_DIR/f'{name.replace(" ", "_")}_seed_predictions_prefix{cutoff}.csv', index=False)
    return dict(mean=mean, seeds=seed_predictions, operands=operands, setting=setting)

# Experiment 1: change only the number of coordinate operands from three to five.
_s8_five_stages, _s8_five_operands = {}, {}
for _s8_cutoff in [21, 28]:
    _s8_rank = pd.read_csv(OUTPUT_DIR/f'coordinate_ranking_prefix{_s8_cutoff}.csv',
                           float_precision='round_trip')
    _s8_chosen = _s8_rank[_s8_rank.eligible].sort_values(
        ['gain', 'original_order'], ascending=[False, True]).head(5).feature.tolist()
    assert len(_s8_chosen) == 5
    assert _s8_chosen[:3] == stages[_s8_cutoff]['operands']
    _s8_five_operands[str(_s8_cutoff)] = _s8_chosen
    _s8_five_stages[_s8_cutoff] = _s8_outer_scores(
        _s8_cutoff, _s8_chosen, 'C0', 'Five operands')
save_json(STEP8_DIR/'five_operand_choices.json', _s8_five_operands)

# Experiment 2: inner chronological selection, followed by fresh outer refits.
_s8_index = pd.Index(train.txId)
_s8_src = _s8_index.get_indexer(edges.txId1)
_s8_dst = _s8_index.get_indexer(edges.txId2)
_s8_keep = (_s8_src >= 0) & (_s8_dst >= 0)
_s8_src, _s8_dst = _s8_src[_s8_keep], _s8_dst[_s8_keep]
_s8_periods = train.time_step.to_numpy()
assert np.array_equal(_s8_periods[_s8_src], _s8_periods[_s8_dst])

def _s8_inner_graph(z, mask):
    keep = mask[_s8_src] & mask[_s8_dst]
    a = csr_matrix((np.ones(int(keep.sum())), (_s8_src[keep], _s8_dst[keep])),
                   shape=(len(train), len(train)))
    incoming, outgoing = a.T.tocsr(), a
    blocks = []
    # Preserve the vectorized summation of the original nested-tuning experiment.
    for direction in [incoming, outgoing]:
        degree = np.asarray(direction.sum(axis=1)).ravel()
        denominator = np.maximum(degree, 1)[:, None]
        mean = np.asarray(direction @ z) / denominator
        variance = np.maximum(np.asarray(direction @ (z*z)) / denominator - mean*mean, 0)
        spread = np.sqrt(variance.mean(axis=1))
        deviation = np.sqrt(((z-mean)**2).mean(axis=1)) / (1+spread)
        spread[degree == 0], deviation[degree == 0] = np.nan, np.nan
        blocks.extend([spread, deviation, (degree > 0).astype(float)])
    return np.column_stack(blocks), incoming, outgoing

_s8_inner_metrics, _s8_scopes, _s8_selected = [], {}, {}
for _s8_cutoff in [21, 28]:
    _s8_end = _s8_cutoff - 7
    _s8_fitting = confirmed[confirmed.time_step <= _s8_end].copy()
    _s8_assessment = confirmed[confirmed.time_step.between(_s8_end+1, _s8_cutoff)].copy()
    _s8_ids, _s8_vids = _s8_fitting.txId.tolist(), _s8_assessment.txId.tolist()
    _s8_y = _s8_assessment.label.to_numpy(dtype=int)
    assert _s8_fitting.label.eq(1).sum() >= 100 and min(np.bincount(_s8_y)) >= 20
    assert _s8_fitting.time_step.max() < _s8_assessment.time_step.min()
    assert _s8_assessment.time_step.max() <= _s8_cutoff
    _s8_geo = fit_geometry(_s8_fitting)
    _s8_z = normalized_features(train, _s8_geo)
    _s8_gf, _s8_ain, _s8_aout = _s8_inner_graph(_s8_z, _s8_periods <= _s8_end)
    _s8_gv, _s8_vin, _s8_vout = _s8_inner_graph(
        _s8_z, (_s8_periods > _s8_end) & (_s8_periods <= _s8_cutoff))
    _s8_fi, _s8_vi = _s8_index.get_indexer(_s8_ids), _s8_index.get_indexer(_s8_vids)
    assert np.all(_s8_gv[_s8_vi][:, [2, 5]].sum(axis=1) > 0), 'Stop: inner isolates require explicit fallback fitting.'
    _s8_f171 = R.loc[_s8_ids, FEATURES].join(
        pd.DataFrame(_s8_gf[_s8_fi], index=_s8_ids, columns=GRAPH_COLUMNS))
    if _s8_end == 21:
        _s8_selector = b0_stages[21]['models'][('GraphSlow', 47)]
    else:
        _s8_selector = _s8_fit(f'inner{_s8_end}_selector', 'Slow200', 47,
                               _s8_f171, _s8_fitting.label, _s8_ids)
    _s8_gain = _s8_selector.booster_.feature_importance(importance_type='gain')[:165]
    _s8_split = 7 if _s8_end == 14 else 14
    _s8_ranking = []
    for _s8_j, _s8_feature in enumerate(FEATURES):
        _s8_values = _s8_fitting[_s8_feature]
        _s8_ks = float(ks_2samp(
            _s8_fitting.loc[_s8_fitting.time_step <= _s8_split, _s8_feature],
            _s8_fitting.loc[_s8_fitting.time_step > _s8_split, _s8_feature],
            method='asymp').statistic)
        _s8_iqr = float(_s8_values.quantile(.75)-_s8_values.quantile(.25))
        _s8_unique = int(_s8_values.nunique())
        _s8_ranking.append(dict(feature=_s8_feature, original_order=_s8_j,
                                gain=float(_s8_gain[_s8_j]), KS=_s8_ks,
                                IQR=_s8_iqr, unique=_s8_unique,
                                eligible=bool(_s8_ks <= .10 and _s8_iqr > 1e-8
                                              and _s8_unique >= 5 and _s8_gain[_s8_j] > 0)))
    _s8_ranking = pd.DataFrame(_s8_ranking)
    _s8_operands = _s8_ranking[_s8_ranking.eligible].sort_values(
        ['gain', 'original_order'], ascending=[False, True]).head(3).feature.tolist()
    assert len(_s8_operands) == 3
    _s8_ranking.to_csv(STEP8_DIR/f'inner{_s8_end}_operand_ranking.csv', index=False)
    _s8_positions = [FEATURES.index(f) for f in _s8_operands]
    _s8_cf, _s8_columns = coordinate_graph_features(
        _s8_z[:, _s8_positions], _s8_ain, _s8_aout, _s8_operands)
    _s8_cv, _ = coordinate_graph_features(
        _s8_z[:, _s8_positions], _s8_vin, _s8_vout, _s8_operands)
    _s8_xf = R.loc[_s8_ids, FEATURES].join(pd.DataFrame(
        np.column_stack([_s8_gf[_s8_fi], _s8_cf[_s8_fi]]),
        index=_s8_ids, columns=GRAPH_COLUMNS+_s8_columns))
    _s8_xv = R.loc[_s8_vids, FEATURES].join(pd.DataFrame(
        np.column_stack([_s8_gv[_s8_vi], _s8_cv[_s8_vi]]),
        index=_s8_vids, columns=GRAPH_COLUMNS+_s8_columns))
    if _s8_end == 21:
        _s8_h1 = raw_stages[21]['models'][('H1', 47)]
    else:
        _s8_h1 = _s8_fit(f'inner{_s8_end}_H1', 'H1', 47,
                         HGB_TRAIN_STANDARD.loc[_s8_ids, FEATURES], _s8_fitting.label, _s8_ids)
    _s8_hp = probability(_s8_h1, HGB_TRAIN_STANDARD.loc[_s8_vids, FEATURES])
    _s8_inner_predictions = _s8_assessment[['txId', 'time_step', 'label']].copy()
    for _s8_setting, _s8_changes in _s8_settings.items():
        _s8_p = .2*_s8_hp
        for _s8_family in ['Slow200', 'RF80']:
            _s8_model = _s8_fit(f'inner{_s8_end}_{_s8_setting}_{_s8_family}',
                                _s8_family, 47, _s8_xf, _s8_fitting.label, _s8_ids, _s8_changes)
            _s8_p = _s8_p + .4*probability(_s8_model, _s8_xv)
        _s8_inner_predictions[_s8_setting] = _s8_p
        _s8_inner_metrics.append(dict(cutoff=_s8_cutoff, setting=_s8_setting,
                                      auc=float(roc_auc_score(_s8_y, _s8_p)),
                                      ap=float(average_precision_score(_s8_y, _s8_p))))
    _s8_order = sorted([r for r in _s8_inner_metrics if r['cutoff'] == _s8_cutoff],
                        key=lambda r: (-r['auc'], -r['ap'], r['setting'] != 'C0', r['setting']))
    _s8_pick = _s8_order[0]['setting']
    _s8_selected[str(_s8_cutoff)] = _s8_pick
    _s8_scopes[str(_s8_cutoff)] = dict(fit_ids=_s8_ids, assessment_ids=_s8_vids,
                                      fit_end=_s8_end, operands=_s8_operands,
                                      geometry=_s8_geo, selected_setting=_s8_pick)
    _s8_inner_predictions.to_csv(STEP8_DIR/f'inner_predictions_prefix{_s8_cutoff}.csv', index=False)

pd.DataFrame(_s8_inner_metrics).to_csv(STEP8_DIR/'inner_metrics.csv', index=False)
save_json(STEP8_DIR/'inner_scopes.json', _s8_scopes)
save_json(STEP8_DIR/'selected_settings.json', _s8_selected)
_s8_tuned_stages = {}
for _s8_cutoff in [21, 28]:
    _s8_tuned_stages[_s8_cutoff] = _s8_outer_scores(
        _s8_cutoff, stages[_s8_cutoff]['operands'], _s8_selected[str(_s8_cutoff)],
        'Nested regularization')

# Calculate every displayed result from out-of-sample prediction probabilities.
_s8_results, _s8_period_results, _s8_seed_results = [], [], []
for _s8_window, _s8_cutoff, _s8_low, _s8_high in WINDOWS:
    for _s8_name, _s8_family_stages, _s8_score in [
        ('Coord3 retained', stages, 'Coord3'),
        ('Five operands', _s8_five_stages, 'score'),
        ('Nested regularization', _s8_tuned_stages, 'score')]:
        _s8_group = _s8_family_stages[_s8_cutoff]['mean']
        _s8_group = _s8_group[_s8_group.time_step.between(_s8_low, _s8_high)
                            & _s8_group.label.isin([0, 1])]
        _s8_result = metrics(_s8_group.label, _s8_group[_s8_score])
        _s8_results.append(dict(window=_s8_window, model=_s8_name, **_s8_result))
        for _s8_period, _s8_part in _s8_group.groupby('time_step'):
            _s8_period_results.append(dict(window=_s8_window, model=_s8_name,
                                           time_step=int(_s8_period),
                                           **metrics(_s8_part.label, _s8_part[_s8_score])))
        if _s8_name != 'Coord3 retained':
            _s8_predictions = _s8_family_stages[_s8_cutoff]['seeds']
            _s8_base = stages[_s8_cutoff]['seeds']
            for _s8_seed in SEEDS:
                _s8_ids = _s8_group.txId.tolist()
                _s8_a = _s8_predictions[_s8_predictions.seed == _s8_seed].set_index('txId').loc[_s8_ids]
                _s8_b = _s8_base[_s8_base.seed == _s8_seed].set_index('txId').loc[_s8_ids]
                assert np.array_equal(_s8_a[['label', 'time_step']], _s8_b[['label', 'time_step']])
                _s8_seed_results.append(dict(window=_s8_window, model=_s8_name, seed=_s8_seed,
                                             auc_change=float(roc_auc_score(_s8_a.label, _s8_a.score)
                                                              -roc_auc_score(_s8_b.label, _s8_b.Coord3))))
_s8_results = pd.DataFrame(_s8_results)
_s8_results.to_csv(STEP8_DIR/'outer_metrics.csv', index=False)
pd.DataFrame(_s8_period_results).to_csv(STEP8_DIR/'period_metrics.csv', index=False)
pd.DataFrame(_s8_seed_results).to_csv(STEP8_DIR/'seed_differences.csv', index=False)
history = _s8_results.pivot(index='model', columns='window', values='auc').rename_axis(columns=None)
history['Long_AP'] = _s8_results[_s8_results.window == 'Long'].set_index('model').ap
history = history.reindex(['Coord3 retained', 'Five operands', 'Nested regularization']).reset_index()
display(pd.DataFrame(_s8_inner_metrics))
display(pd.DataFrame([dict(cutoff=int(k), selected_setting=v) for k, v in _s8_selected.items()]))
display(history)
display(_s8_results[['window', 'model', 'tp', 'fp', 'fn', 'recall', 'precision']])
print('New fits in Step 8:', _s8_new_fits)
print('Retained final recipes are unchanged; all displayed metrics were calculated from predictions.')


,cutoff,setting,auc,ap
0,21,C0,0.992640,0.980163
1,21,L2,0.992477,0.980389
2,21,Leaves15,0.993016,0.980723
3,28,C0,0.954796,0.883770
4,28,L2,0.956074,0.882984
5,28,Leaves15,0.948301,0.877327


,cutoff,selected_setting
0,21,Leaves15
1,28,L2


,model,A,B,Long,Long_AP
0,Coord3 retained,0.957081,0.996957,0.869111,0.775802
1,Five operands,0.955791,0.996824,0.861414,0.770938
2,Nested regularization,0.957416,0.996515,0.859334,0.768773


,window,model,tp,fp,fn,recall,precision
0,A,Coord3 retained,526,27,145,0.783905,0.951175
1,A,Five operands,527,23,144,0.785395,0.958182
2,A,Nested regularization,527,33,144,0.785395,0.941071
3,B,Coord3 retained,998,12,104,0.905626,0.988119
4,B,Five operands,1004,15,98,0.911071,0.985280
5,B,Nested regularization,1002,19,100,0.909256,0.981391
6,Long,Coord3 retained,675,25,427,0.612523,0.964286
7,Long,Five operands,671,25,431,0.608893,0.964080
8,Long,Nested regularization,665,37,437,0.603448,0.947293


New fits in Step 8: 0
Retained final recipes are unchanged; all displayed metrics were calculated from predictions.


## 9. Check temporal consistency and limits

I next inspect per-period Long AUC/AP and paired seed changes. This asks whether the pooled improvement is broad or depends on a few seeds/periods. Coord3's Long gain occurs in two of three seeds, so it is not uniformly stable.

These models preserve the observed confirmed-label imbalance without weighting/oversampling. AP and recall/precision complement AUC. The0.5 threshold remains fixed, not tuned on outer labels. Reused windows, dependent graph nodes and absent natural historical isolates limit inference; exact fallback tests do not establish isolated test-population accuracy. Graph features are associations, not causal evidence of fraud.


In [35]:
long_periods = per_period[(per_period.window=='Long')&per_period.model.isin(['B0','Coord3'])]
display(long_periods.pivot(index='time_step',columns='model',values=['auc','ap']))
seed_rows = []
for window,cutoff,low,high in [('A',21,22,28),('B',28,29,35),('Long',21,29,35)]:
    q = stages[cutoff]['seeds']
    q = q[q.time_step.between(low,high)&q.label.isin([0,1])]
    for seed,g in q.groupby('seed'):
        seed_rows.append({'window':window,'seed':int(seed),
                          'delta_auc_Coord3_minus_B0':roc_auc_score(g.label,g.Coord3)-roc_auc_score(g.label,g.B0),
                          'delta_ap_Coord3_minus_B0':average_precision_score(g.label,g.Coord3)-average_precision_score(g.label,g.B0)})
seed_rows = pd.DataFrame(seed_rows)
seed_rows.to_csv(OUTPUT_DIR/'paired_seed_deltas.csv',index=False)
display(seed_rows[seed_rows.window=='Long'])
print('No validation threshold is selected; confusion matrices use the fixed threshold0.5.')


auc                  ap          
model            B0    Coord3        B0    Coord3
time_step                                        
29         0.743753  0.755040  0.640408  0.647028
30         0.990820  0.991749  0.960262  0.965880
31         0.774678  0.790126  0.582725  0.589755
32         0.801193  0.819596  0.792678  0.799088
33         0.993447  0.994799  0.931535  0.939506
34         0.997908  0.997851  0.974630  0.974029
35         0.996283  0.996596  0.983323  0.984905

,window,seed,delta_auc_Coord3_minus_B0,delta_ap_Coord3_minus_B0
6,Long,47,0.008204,0.005202
7,Long,48,-0.000929,0.002625
8,Long,49,0.031940,0.014663


No validation threshold is selected; confusion matrices use the fixed threshold0.5.


## 10. Train both final recipes and generate their submission files

Once the recipes are retained, fit on all 31,235 confirmed transactions in periods 1–35.

Predict illicit-class probabilities, preserve the sample's `index,target` ordering, verify all rows/ranges/readback, and save the copies。 


In [36]:
final_stage = fit_stage(35,test,final=True)
final_mean = final_stage['mean'].set_index('txId').loc[test.txId]
print('Final training-only operands:', final_stage['operands'])
print('Test isolates:',int(final_mean.isolated.sum()),'of',len(final_mean))
SUBMISSION_DIR = OUTPUT_DIR / 'submissions'
SUBMISSION_DIR.mkdir(exist_ok=True)
integrity, summary = [], []
for model, filename in [('B0','graph_lgbm_hgb_reproduced.csv'),('Coord3','graph_lgbm_hgb_coord3_reproduced.csv')]:
    score = final_mean[model].to_numpy()
    out = sample.copy(); out['target'] = score
    path = SUBMISSION_DIR / filename
    out.to_csv(path,index=False)
    back = pd.read_csv(path,float_precision='round_trip')
    checks = {
        '15329_rows':len(back)==15329,
        'columns_index_target':back.columns.tolist()==['index','target'],
        'template_order':np.array_equal(back['index'],sample['index']),
        'unique_nonmissing_index':back['index'].is_unique and back['index'].notna().all(),
        'finite_probability':bool(np.isfinite(back.target).all()),
        'range_0_1':bool(back.target.between(0,1).all()),
        'CSV_roundtrip':np.array_equal(back.target,score),
        'class1_orientation':True,
    }
    assert all(checks.values())
    integrity.extend({'model':model,'check':k,'passed':bool(v)} for k,v in checks.items())
    summary.append({'model':model,'CSV':str(path.relative_to(ROOT)),'sha256':file_hash(path),
                    'min':float(score.min()),'max':float(score.max()),'mean':float(score.mean()),
                    'median':float(np.median(score)),'share_above_half':float(np.mean(score>.5)),
                    'unique_probabilities':int(np.unique(score).size)})
mask = final_mean.isolated.to_numpy()
assert np.array_equal(final_mean.loc[mask,'B0'],final_mean.loc[mask,'Coord3'])
pd.DataFrame(integrity).to_csv(OUTPUT_DIR/'submission_integrity.csv',index=False)
save_json(OUTPUT_DIR/'submission_summary.json',summary)
display(pd.DataFrame(summary)[['model','CSV','mean','min','max']])


Final training-only operands: ['feat_163', 'feat_55', 'feat_59']
Test isolates: 3891 of 15329


,model,CSV,mean,min,max
0,B0,reproducibility/final_two_models/submissions/g...,0.045767,0.000013,0.999671
1,Coord3,reproducibility/final_two_models/submissions/g...,0.046006,0.000014,0.999671


## Final development outcome

The development path was **raw-feature references → controlled configuration changes → raw probability ensemble → six-feature graph B0 → three-coordinate Coord3 → rejected extensions → generate submission files**.

B0 provides neighbourhood context while keeping raw H1 and exact raw fallback. Coord3 retains that design and exposes coordinate-specific spread/mismatch. Both are selected from measured matched development comparisons, not from a claim that every component wins independently. The public submission outcomes were 0.94878 and 0.94958.
